<a href="https://colab.research.google.com/github/kuds/rl-doom/blob/main/notebooks/03_ppo_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — PPO Training (Stable-Baselines3)

Train a **PPO** agent (via `stable_baselines3.PPO`, `CnnPolicy`) on four ViZDoom
scenarios: *Basic*, *Deadly Corridor*, *Defend the Center*, and *Deathmatch*.

**Config-driven.** All hyperparameters, env settings, and training budgets
come from `configs/ppo_<scenario>.yaml` so the notebook and the standalone
YAML files can never drift apart. Tune a scenario by editing its YAML; the
notebook just iterates over the configs and calls `train_sb3`.

**Design notes.** Each scenario is trained end-to-end and its full artifact
bundle (learning curves, eval curve, video, checkpoint, stage summary) is
written to disk **before** the next scenario starts. That way a Colab timeout
or an error mid-notebook still leaves every completed scenario shippable.

**Colab baseline.** Defaults target an **L4 GPU + high-memory runtime**:
`n_envs=8` with `DummyVecEnv` (in-process; no `SubprocVecEnv` — ViZDoom +
Colab don't play well with Python subprocess workers), `n_steps=1024` per
env, `batch_size=256`.

## 1. Setup

In [ ]:
# --- Colab Setup ---
# Uncomment the block below when running on Google Colab
import subprocess, os
if not os.path.exists("/content/rl-doom"):
    subprocess.run(["git", "clone", "https://github.com/kuds/rl-doom.git", "/content/rl-doom"], check=True)
os.chdir("/content/rl-doom/notebooks")
subprocess.run(["pip", "install", "-q", "-e", "/content/rl-doom[notebooks]"], check=True)

import sys
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import torch

from rl_doom.paths import load_yaml_config, new_run_dir, write_config
from rl_doom.sb3_utils import gpu_info, policy_kwargs_from_config, train_sb3

# L4 benefits noticeably from cuDNN autotuning on fixed-shape CNNs.
torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Global RNG seeding for notebook-level reproducibility. SB3 seeds
# itself per-run via the ``seed=`` kwarg we forward into ``train_sb3``;
# this block covers non-SB3 randomness (numpy, Python ``random``,
# torch) that the notebook may consume outside the training loop.
import random as _random
GLOBAL_SEED = 42
_random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
GPU_INFO = gpu_info()
print(f"Using device: {DEVICE}")
for k, v in GPU_INFO.items():
    print(f"  {k}: {v}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the block below when running on Google Colab
# ---
import shutil
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Finding Theta/rl-doom"
os.makedirs(DRIVE_ROOT, exist_ok=True)
for subdir in ["training_jobs", "analysis"]:
    drive_dir = f"{DRIVE_ROOT}/{subdir}"
    local_dir = os.path.abspath(f"../{subdir}")
    os.makedirs(drive_dir, exist_ok=True)
    if os.path.islink(local_dir):
        os.remove(local_dir)
    if os.path.isdir(local_dir):
        for f in os.listdir(local_dir):
            src = os.path.join(local_dir, f)
            dst = os.path.join(drive_dir, f)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(local_dir)
    os.symlink(drive_dir, local_dir)
print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")
# ---

## 2. Load per-scenario configs

Each scenario's hyperparameters, env settings, and training budget live
in `configs/ppo_<scenario>.yaml`. We load all four upfront so the loop
below can iterate without touching the YAML files again.

In [ ]:
SCENARIOS = ["basic", "deadly_corridor", "defend_the_center", "deathmatch"]

CONFIGS = {s: load_yaml_config(f"ppo_{s}") for s in SCENARIOS}

# Sanity dump of the knobs that will drive training.
for s, cfg in CONFIGS.items():
    hp = cfg["hyperparams"]
    env_cfg = cfg.get("env", {})
    training = cfg["training"]
    print(
        f"{s:20s}  total_timesteps={training['total_timesteps']:>10,}  "
        f"n_envs={env_cfg.get('n_envs', 1):>2}  "
        f"lr={hp['lr']:.1e}  ent_coef={hp['entropy_coef']:.3f}"
    )

## 3. Train each scenario (artifacts written per scenario)

`train_sb3` does the full end-to-end training for one scenario:
1. Builds a `DummyVecEnv` wrapped in SB3's `Monitor` for episode logging.
2. Calls `PPO.learn()` with `EvalCallback` + `CheckpointCallback`.
3. On completion: writes `training.npz`, `learning_curves.png`,
   `eval_performance.png`, the final video, `stage_summary.txt`, and
   updates the `latest` pointer.

The loop pulls hyperparams, env settings (resize/frame_skip/num_stack/
doom_skill/num_bots), policy kwargs, seed, and eval/checkpoint schedule
from the YAML. `config.json` records the full merged view so each run
is reproducible from a single file.

In [ ]:
from IPython.display import Video, display

results = {}
for scenario in SCENARIOS:
    cfg = CONFIGS[scenario]
    hp = dict(cfg["hyperparams"])
    env_cfg = dict(cfg.get("env", {}))
    policy_cfg = dict(cfg.get("policy", {}))
    training_cfg = cfg["training"]
    eval_cfg = cfg.get("eval", {})
    seed = int(cfg.get("seed", 42))
    total_ts = int(training_cfg["total_timesteps"])
    n_envs = int(env_cfg.get("n_envs", 8))

    print("\n" + "=" * 70)
    print(f"[PPO] {scenario}  |  total_timesteps={total_ts:,}  |  n_envs={n_envs}  |  seed={seed}")
    print("=" * 70)

    run_dir = new_run_dir(scenario, "ppo", seed=seed)
    write_config(
        run_dir,
        env=scenario,
        algo="ppo",
        seed=seed,
        hyperparams={**hp, "total_timesteps": total_ts, "n_envs": n_envs},
        env_settings=env_cfg,
        policy_kwargs=policy_cfg,
        gpu_setup=GPU_INFO,
        training_schedule={
            "checkpoint_freq": int(training_cfg.get("checkpoint_freq", 100_000)),
            "eval_freq": int(eval_cfg.get("eval_freq", 25_000)),
            "eval_episodes": int(eval_cfg.get("n_episodes", 10)),
        },
    )

    def _show(rd, scenario=scenario):
        vids = sorted((rd / "media").glob(f"ppo_{scenario}.*"))
        if vids:
            print(f"[video] {vids[0]}")
            try:
                display(Video(str(vids[0]), embed=True))
            except Exception as exc:
                print(f"  (inline display skipped: {exc})")

    result = train_sb3(
        algo="ppo",
        scenario=scenario,
        run_dir=run_dir,
        hyperparams=hp,
        seed=seed,
        total_timesteps=total_ts,
        n_envs=n_envs,
        eval_freq=int(eval_cfg.get("eval_freq", 25_000)),
        eval_episodes=int(eval_cfg.get("n_episodes", 10)),
        checkpoint_freq=int(training_cfg.get("checkpoint_freq", 100_000)),
        record_video=True,
        device=DEVICE,
        on_complete=_show,
        resize_shape=tuple(env_cfg.get("resize_shape", (84, 84))),
        frame_skip=int(env_cfg.get("frame_skip", 4)),
        num_stack=int(env_cfg.get("num_stack", 4)),
        doom_skill=env_cfg.get("doom_skill"),
        num_bots=int(env_cfg.get("num_bots", 0)),
        policy_kwargs=policy_kwargs_from_config(policy_cfg),
    )
    results[scenario] = result
    print(
        f"[done] {scenario}: wall={result['wall_time_seconds']:.1f}s | "
        f"fps={result['fps']:.0f} | eval_mean={result['mean_eval_reward']}"
    )

print("\nAll PPO scenarios complete.")
for s, r in results.items():
    print(f"  - {s}: {r['run_dir']}")

## 4. Cross-scenario eval summary

Each run already wrote its own `learning_curves.png` and `eval_performance.png`
to `run_dir/figures/`. The cell below just reloads the per-run `training.npz`
files to compare evaluation curves on one axis.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure(figsize=(10, 5))
for scenario, result in results.items():
    npz_path = Path(result["run_dir"]) / "metrics" / "training.npz"
    if not npz_path.exists():
        continue
    data = np.load(npz_path)
    eval_log = data["eval_rewards"]
    if eval_log.ndim == 2 and eval_log.shape[0] > 0:
        steps = eval_log[:, 0]
        means = eval_log[:, 1]
        stds = eval_log[:, 2]
        plt.plot(steps, means, marker="o", label=scenario)
        plt.fill_between(steps, means - stds, means + stds, alpha=0.2)
plt.xlabel("Environment Steps")
plt.ylabel("Eval Reward")
plt.title("PPO — Cross-scenario eval")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Disconnect runtime

In [ ]:
# Disconnect the Colab runtime at the end of the notebook to save compute.
# No-op when running locally.
try:
    from google.colab import runtime as _colab_runtime
except ImportError:
    pass
else:
    import time
    print("Notebook finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    _colab_runtime.unassign()